In [1]:
import os
os.environ["SYNOPTIC_TOKEN"] = "b1ab132d60f04cb4b6eb3e7d9f31699b"

In [2]:
import os, requests, json

token = os.getenv("SYNOPTIC_TOKEN","").strip()
url = "https://api.synopticdata.com/v2/stations/metadata"
params = {
    "state": "ID",
    "status": "active",
    "token": token,
}

r = requests.get(url, params=params, timeout=60)
print("HTTP:", r.status_code)
js = r.json()
print("Keys:", js.keys())
print(json.dumps(js.get("SUMMARY", {}), indent=2))

HTTP: 200
Keys: dict_keys(['STATION', 'SUMMARY'])
{
  "NUMBER_OF_OBJECTS": 1175,
  "RESPONSE_CODE": 1,
  "RESPONSE_MESSAGE": "OK",
  "METADATA_QUERY_TIME": "169.4 ms",
  "METADATA_PARSE_TIME": "7.1 ms",
  "TOTAL_METADATA_TIME": "176.6 ms",
  "TOTAL_TIME": "176.6 ms",
  "VERSION": "v2.31.0"
}


In [3]:
#!/usr/bin/env python3
"""
Idaho HydroMet Stations Ingest + Cross-Match
- SNOTEL (NRCS/WCC) Idaho station list
- Synoptic (Weather API) Idaho station metadata
- UI/UNI stations from local file (CSV/Excel) (template)
- Match shared stations via geographic proximity (+ optional fuzzy names)

Outputs:
- out/stations_all.csv
- out/matches_pairs.csv
- out/overlap_summary.csv
"""

from __future__ import annotations

import os
import re
import math
import json
import time
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

import requests
import pandas as pd


# -----------------------------
# CONFIG
# -----------------------------

SNOTEL_ID_URL = "https://wcc.sc.egov.usda.gov/nwcc/sntlsites.jsp?state=ID"
SYNOPTIC_METADATA_URL = "https://api.synopticdata.com/v2/stations/metadata"

# Put your UI/UNI station list here (you can export from any spreadsheet)
UI_STATIONS_PATH = "data/ui_stations.csv"  # or .xlsx

OUTPUT_DIR = "out"

# Match threshold (meters)
MAX_DIST_M = 200.0

# If you want fuzzy name matching, set True and install rapidfuzz
USE_FUZZY_NAME = False
FUZZY_MIN_SCORE = 92  # 0-100


# -----------------------------
# HELPERS
# -----------------------------

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def norm_name(s: str) -> str:
    """Normalize station names for matching."""
    s = clean_text(s).lower()
    s = re.sub(r"[^a-z0-9 ]+", "", s)
    s = re.sub(r"\b(station|stn|site)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in meters."""
    R = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2.0)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


# -----------------------------
# INGEST: SNOTEL
# -----------------------------
def fetch_snotel_idaho(timeout: int = 30) -> pd.DataFrame:
    """
    Pull SNOTEL Idaho station list from NRCS/WCC page and return a dataframe.

    IMPORTANT:
    - The current HTML table for ID does NOT include a dedicated numeric Site ID column.
      The numeric site id is embedded in the site_name string like: "Couch Summit (1306)".
    - We therefore extract the site id from parentheses and use it as source_station_id.
    """
    print(f"[SNOTEL] Fetching: {SNOTEL_ID_URL}")

    # Read all tables from the page
    tables = pd.read_html(SNOTEL_ID_URL)
    if not tables:
        raise RuntimeError("No tables found on SNOTEL page. Page layout may have changed.")

    # Choose the largest table (usually the stations listing)
    df = max(tables, key=lambda t: t.shape[0]).copy()

    # Normalize column names
    df.columns = [clean_text(c).lower() for c in df.columns]
    print("SNOTEL columns:", df.columns.tolist())

    # Expected columns from your run:
    # ['ntwk', 'state', 'site_name', 'ts', 'start', 'lat', 'lon', 'elev', 'county', 'huc']
    required = ["site_name", "lat", "lon"]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise RuntimeError(f"Missing expected columns {missing_cols}. Got: {df.columns.tolist()}")

    # Build output with extra context columns (kept if present)
    out = pd.DataFrame({
        "source": "SNOTEL",
        "station_name_raw": df["site_name"].astype(str),
        "latitude": pd.to_numeric(df["lat"], errors="coerce"),
        "longitude": pd.to_numeric(df["lon"], errors="coerce"),
        "elevation_m": pd.to_numeric(df["elev"], errors="coerce") if "elev" in df.columns else pd.NA,
        "county": df["county"].astype(str) if "county" in df.columns else pd.NA,
        "huc": df["huc"].astype(str) if "huc" in df.columns else pd.NA,
        "network_code": df["ntwk"].astype(str) if "ntwk" in df.columns else pd.NA,
        "timeseries_code": df["ts"].astype(str) if "ts" in df.columns else pd.NA,
        "start": df["start"].astype(str) if "start" in df.columns else pd.NA,
    })

    # Clean raw station name
    out["station_name_raw"] = out["station_name_raw"].map(clean_text)

    # Extract numeric site_id from "Name (1234)" in station_name_raw
    out["source_station_id"] = out["station_name_raw"].str.extract(r"\((\d+)\)", expand=False)

    # Clean station_name (remove trailing "(####)")
    out["station_name"] = (
        out["station_name_raw"]
        .str.replace(r"\s*\(\d+\)\s*", "", regex=True)
        .str.strip()
    )

    # Normalize station name for matching
    out["name_norm"] = out["station_name"].map(norm_name)

    # Stable uid
    out["station_uid"] = "SNOTEL:" + out["source_station_id"].astype(str)

    # Drop rows missing coordinates
    out = out.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

    # Safety checks
    missing_id = out["source_station_id"].isna().sum()
    if missing_id:
        print(f"[SNOTEL] WARNING: {missing_id} stations missing site_id extraction")

    # Keep only the standard columns expected downstream + a few useful extras
    keep_cols = [
        "source",
        "source_station_id",
        "station_name",
        "latitude",
        "longitude",
        "elevation_m",
        "county",
        "huc",
        "network_code",
        "name_norm",
        "station_uid",
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[keep_cols].copy()

    print(f"[SNOTEL] Stations loaded: {len(out)}")
    return out

# -----------------------------
# INGEST: Synoptic
# -----------------------------
def fetch_synoptic_idaho(token: str, timeout: int = 60) -> pd.DataFrame:
    if not token:
        raise ValueError("Synoptic token missing. Set env var SYNOPTIC_TOKEN.")

    params = {
        "state": "ID",
        "status": "active",
        "token": token,
    }

    print("[Synoptic] Requesting metadata for Idaho (active)")
    resp = requests.get(SYNOPTIC_METADATA_URL, params=params, timeout=timeout)
    resp.raise_for_status()
    js = resp.json()

    if "STATION" not in js:
        raise RuntimeError(f"Unexpected Synoptic response keys: {list(js.keys())} | SUMMARY={js.get('SUMMARY')}")

    rows = []
    for s in js["STATION"]:
        stid = clean_text(s.get("STID") or s.get("stid"))
        name = clean_text(s.get("NAME") or s.get("name"))
        lat  = s.get("LATITUDE") or s.get("latitude")
        lon  = s.get("LONGITUDE") or s.get("longitude")
        elev = s.get("ELEVATION") or s.get("elevation")

        # ✅ Network ID in Synoptic metadata is usually MNET_ID
        mnet_id = clean_text(s.get("MNET_ID") or s.get("mnet_id") or "")

        # optional extras if present in this “light” metadata call
        county = clean_text(s.get("COUNTY") or s.get("county") or "")

        rows.append({
            "source": "SYNOPTIC",
            "source_station_id": stid,
            "station_name": name,
            "latitude": pd.to_numeric(lat, errors="coerce"),
            "longitude": pd.to_numeric(lon, errors="coerce"),
            "elevation_m": pd.to_numeric(elev, errors="coerce"),
            "network_id": mnet_id,   # keep column name "network_id" for downstream
            "county": county if county else None,
        })

    out = pd.DataFrame(rows)
    out["name_norm"] = out["station_name"].map(norm_name)
    out["station_uid"] = "SYNOPTIC:" + out["source_station_id"].astype(str)
    out = out.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

    print(f"[Synoptic] Stations loaded: {len(out)}")
    return out


# -----------------------------
# INGEST: UI/UNI (local file template)
# -----------------------------

def load_ui_stations(path: str) -> pd.DataFrame:
    """
    Load UI/UNI station list from local file.

    Expected columns (minimum):
      - station_name
      - latitude
      - longitude

    Optional:
      - station_code or source_station_id
      - elevation_m
      - network_name (e.g., 'UI', 'Idaho Mesonet', 'Agriculture station list')
    """
    if not os.path.exists(path):
        print(f"[UI] File not found: {path} (skip). Put your list there when ready.")
        return pd.DataFrame(columns=[
            "source","source_station_id","station_name","latitude","longitude","elevation_m","network_name","name_norm","station_uid"
        ])

    print(f"[UI] Loading: {path}")
    if path.lower().endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    # Normalize column names
    df.columns = [clean_text(c) for c in df.columns]
    cols_lower = {c.lower(): c for c in df.columns}

    def get_col(*cands):
        for cand in cands:
            if cand.lower() in cols_lower:
                return cols_lower[cand.lower()]
        return None

    c_name = get_col("station_name", "name", "site_name")
    c_lat = get_col("latitude", "lat")
    c_lon = get_col("longitude", "lon", "lng")
    c_id  = get_col("source_station_id", "station_code", "station_id", "id")
    c_elev = get_col("elevation_m", "elevation", "elev_m", "elev")
    c_net = get_col("network_name", "network", "owner")

    out = pd.DataFrame({
        "source": "UI",
        "source_station_id": df[c_id].astype(str) if c_id else None,
        "station_name": df[c_name].astype(str) if c_name else None,
        "latitude": pd.to_numeric(df[c_lat], errors="coerce") if c_lat else None,
        "longitude": pd.to_numeric(df[c_lon], errors="coerce") if c_lon else None,
        "elevation_m": pd.to_numeric(df[c_elev], errors="coerce") if c_elev else None,
        "network_name": df[c_net].astype(str) if c_net else "University of Idaho",
    })

    out["station_name"] = out["station_name"].map(clean_text)
    out["name_norm"] = out["station_name"].map(norm_name)

    # station_uid: prefer code if present, else hash-like from name+coords
    def make_uid(r):
        sid = clean_text(r.get("source_station_id", ""))
        if sid:
            return f"UI:{sid}"
        # fallback
        return f"UI:{r['name_norm']}:{round(r['latitude'],5)}:{round(r['longitude'],5)}"
    out["station_uid"] = out.apply(make_uid, axis=1)

    out = out.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    print(f"[UI] Stations loaded: {len(out)}")
    return out


# -----------------------------
# MATCHING
# -----------------------------

def pairwise_match_by_distance(
    df_a: pd.DataFrame,
    df_b: pd.DataFrame,
    max_dist_m: float = MAX_DIST_M,
    label_a: str = "A",
    label_b: str = "B",
) -> pd.DataFrame:
    """
    Brute-force pairwise distance matching (fine for few thousand points).
    If Synoptic is large, we can upgrade to BallTree later.
    """
    matches = []
    a = df_a.reset_index(drop=True)
    b = df_b.reset_index(drop=True)

    for i, ra in a.iterrows():
        lat1, lon1 = float(ra["latitude"]), float(ra["longitude"])
        best_j = None
        best_d = None

        for j, rb in b.iterrows():
            d = haversine_m(lat1, lon1, float(rb["latitude"]), float(rb["longitude"]))
            if d <= max_dist_m and (best_d is None or d < best_d):
                best_d = d
                best_j = j

        if best_j is not None:
            rb = b.loc[best_j]
            matches.append({
                f"{label_a}_uid": ra["station_uid"],
                f"{label_a}_name": ra["station_name"],
                f"{label_a}_source_id": ra["source_station_id"],
                f"{label_a}_source": ra["source"],
                f"{label_b}_uid": rb["station_uid"],
                f"{label_b}_name": rb["station_name"],
                f"{label_b}_source_id": rb["source_station_id"],
                f"{label_b}_source": rb["source"],
                "dist_m": float(best_d),
            })

    return pd.DataFrame(matches).sort_values("dist_m").reset_index(drop=True)


def optional_fuzzy_filter(matches: pd.DataFrame) -> pd.DataFrame:
    """
    Optional: filter matches by fuzzy name similarity to reduce false positives.
    Requires: pip install rapidfuzz
    """
    if matches.empty or not USE_FUZZY_NAME:
        return matches

    try:
        from rapidfuzz import fuzz
    except Exception as e:
        raise RuntimeError("USE_FUZZY_NAME=True but rapidfuzz not installed. Run: pip install rapidfuzz") from e

    scores = []
    for _, r in matches.iterrows():
        a = norm_name(r.filter(like="_name").iloc[0])
        b = norm_name(r.filter(like="_name").iloc[1])
        score = fuzz.token_sort_ratio(a, b)
        scores.append(score)

    out = matches.copy()
    out["name_score"] = scores
    out = out[out["name_score"] >= FUZZY_MIN_SCORE].reset_index(drop=True)
    return out


# -----------------------------
# MAIN
# -----------------------------

def main():
    ensure_dir(OUTPUT_DIR)

    # 1) Load sources
    df_snotel = fetch_snotel_idaho()
    syn_token = os.getenv("SYNOPTIC_TOKEN", "").strip()
    df_syn = pd.DataFrame()
    if syn_token:
        df_syn = fetch_synoptic_idaho(syn_token)
    else:
        print("[Synoptic] SYNOPTIC_TOKEN not set -> skipping Synoptic ingest for now.")

    df_ui = load_ui_stations(UI_STATIONS_PATH)

    # 2) Combine master table
    df_all = pd.concat([df_snotel, df_syn, df_ui], ignore_index=True, sort=False)
    df_all.to_csv(os.path.join(OUTPUT_DIR, "stations_all.csv"), index=False)
    print(f"[OUT] Wrote {OUTPUT_DIR}/stations_all.csv ({len(df_all)} rows)")

    # 3) Match overlaps
    # SNOTEL <-> Synoptic
    if not df_syn.empty:
        m_sn_sy = pairwise_match_by_distance(df_snotel, df_syn, MAX_DIST_M, "SNOTEL", "SYNOPTIC")
        m_sn_sy = optional_fuzzy_filter(m_sn_sy)
        m_sn_sy.to_csv(os.path.join(OUTPUT_DIR, "matches_snotel_synoptic.csv"), index=False)
        print(f"[MATCH] SNOTEL<->Synoptic matches: {len(m_sn_sy)}")

    # UI <-> Synoptic
    if not df_syn.empty and not df_ui.empty:
        m_ui_sy = pairwise_match_by_distance(df_ui, df_syn, MAX_DIST_M, "UI", "SYNOPTIC")
        m_ui_sy = optional_fuzzy_filter(m_ui_sy)
        m_ui_sy.to_csv(os.path.join(OUTPUT_DIR, "matches_ui_synoptic.csv"), index=False)
        print(f"[MATCH] UI<->Synoptic matches: {len(m_ui_sy)}")

    # UI <-> SNOTEL
    if not df_ui.empty:
        m_ui_sn = pairwise_match_by_distance(df_ui, df_snotel, MAX_DIST_M, "UI", "SNOTEL")
        m_ui_sn = optional_fuzzy_filter(m_ui_sn)
        m_ui_sn.to_csv(os.path.join(OUTPUT_DIR, "matches_ui_snotel.csv"), index=False)
        print(f"[MATCH] UI<->SNOTEL matches: {len(m_ui_sn)}")

    # 4) Summary
    summary_rows = []
    for src, df in [("SNOTEL", df_snotel), ("SYNOPTIC", df_syn), ("UI", df_ui)]:
        summary_rows.append({"source": src, "n_stations": int(len(df))})
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(os.path.join(OUTPUT_DIR, "overlap_summary.csv"), index=False)
    print(f"[OUT] Wrote {OUTPUT_DIR}/overlap_summary.csv")

    print("\nDone.")


if __name__ == "__main__":
    main()

[SNOTEL] Fetching: https://wcc.sc.egov.usda.gov/nwcc/sntlsites.jsp?state=ID
SNOTEL columns: ['ntwk', 'state', 'site_name', 'ts', 'start', 'lat', 'lon', 'elev', 'county', 'huc']
[SNOTEL] Stations loaded: 85
[Synoptic] Requesting metadata for Idaho (active)
[Synoptic] Stations loaded: 1175
[UI] File not found: data/ui_stations.csv (skip). Put your list there when ready.
[OUT] Wrote out/stations_all.csv (1260 rows)
[MATCH] SNOTEL<->Synoptic matches: 14
[OUT] Wrote out/overlap_summary.csv

Done.


In [4]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import json
import time
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional

import requests
import pandas as pd

SYNOPTIC_TOKEN = os.getenv("SYNOPTIC_TOKEN", "").strip()

# --- Files produced by your ingest step ---
STATIONS_ALL_CSV = "out/stations_all.csv"
OUT_STATIONS_ENRICHED = "out/stations_enriched.csv"
OUT_VARS = "out/station_variables.csv"
CACHE_DIR = "out/cache_api"
os.makedirs(CACHE_DIR, exist_ok=True)

# --- Synoptic endpoints ---
SYN_STATION_METADATA = "https://api.synopticdata.com/v2/stations/metadata"
SYN_STATION_LATEST   = "https://api.synopticdata.com/v2/stations/latest"   # to infer variable availability via observations
SYN_STATION_TIMESERIES = "https://api.synopticdata.com/v2/stations/timeseries"  # optional (heavy)

# --- SNOTEL endpoints (NRCS/WCC) ---
# We'll use reportGenerator to list data availability for specific elements (robust & public).
SNOTEL_REPORTGEN = "https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/customSingleStationReport"


def utcnow_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def safe_get_json(url: str, params: Dict[str, Any], cache_key: str, timeout: int = 60, retries: int = 3) -> Dict[str, Any]:
    """
    Requests JSON with disk cache and retries. Cache avoids re-hitting API for the same station.
    """
    cache_path = os.path.join(CACHE_DIR, f"{cache_key}.json")
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            return json.load(f)

    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            js = r.json()
            with open(cache_path, "w") as f:
                json.dump(js, f)
            return js
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"GET failed after retries. url={url} params={params} err={last_err}")


# -------------------------
# SYNOPTIC: per-station metadata (sensor / vars)
# -------------------------

def synoptic_station_metadata(stid: str) -> Dict[str, Any]:
    if not SYNOPTIC_TOKEN:
        raise ValueError("SYNOPTIC_TOKEN not set.")
    params = {
        "stid": stid,
        "token": SYNOPTIC_TOKEN,
        # no 'fields' to avoid 'Field not known'
    }
    return safe_get_json(SYN_STATION_METADATA, params, cache_key=f"syn_metadata_{stid}")


def synoptic_station_latest(stid: str) -> Dict[str, Any]:
    """
    Useful to infer what variables are actively reporting in 'OBSERVATIONS' and any metadata there.
    """
    if not SYNOPTIC_TOKEN:
        raise ValueError("SYNOPTIC_TOKEN not set.")
    params = {
        "stid": stid,
        "token": SYNOPTIC_TOKEN,
    }
    return safe_get_json(SYN_STATION_LATEST, params, cache_key=f"syn_latest_{stid}")


def parse_synoptic_variables(stid: str) -> List[Dict[str, Any]]:
    """
    Build long-format variable rows for a Synoptic station.
    Tries:
      A) metadata payload for sensor/variable info (if present)
      B) latest observations keys (what is currently observed)
    """
    meta = synoptic_station_metadata(stid)
    latest = synoptic_station_latest(stid)

    rows: List[Dict[str, Any]] = []
    retrieved = utcnow_iso()

    # 1) Variables from LATEST observations (always present when station returns data)
    # Structure: latest["STATION"][0]["OBSERVATIONS"] has keys like "air_temp_value_1", etc.
    try:
        st0 = latest.get("STATION", [])[0]
        obs = st0.get("OBSERVATIONS", {}) or {}
        units = st0.get("UNITS", {}) or {}
        # Map observation keys -> a normalized variable_code
        # Keep it simple: strip suffix patterns, preserve native key.
        for k in obs.keys():
            # example native keys: air_temp_value_1, wind_speed_value_1, precipitation_accum_one_hour_value_1
            native = k
            base = native.replace("_value_1", "").replace("_value_1d", "")
            unit = units.get(native) or units.get(base) or None
            rows.append({
                "station_uid": f"SYNOPTIC:{stid}",
                "source": "SYNOPTIC",
                "variable_code": base,          # you can later map to a controlled vocabulary
                "variable_native": native,
                "unit": unit,
                "sensor_height_m": None,
                "sensor_depth_m": None,
                "aggregation": None,
                "sampling_interval_s": None,
                "latency_s": None,
                "qaqc_level": None,
                "metadata_retrieved_utc": retrieved,
                "source_details_json": json.dumps({"from": "latest", "example": obs.get(k)}, ensure_ascii=False),
            })
    except Exception:
        pass

    # 2) If METADATA has sensor inventory / variable list, parse it (structure can vary by network)
    # We keep raw block in source_details_json if we find something.
    try:
        st_meta = meta.get("STATION", [])[0]
        # Some payloads include things like "SENSOR_VARIABLES" or similar; if present, serialize.
        for key in st_meta.keys():
            if "SENSOR" in key.upper() or "VARIABLE" in key.upper():
                rows.append({
                    "station_uid": f"SYNOPTIC:{stid}",
                    "source": "SYNOPTIC",
                    "variable_code": f"__meta_block__{key}",
                    "variable_native": key,
                    "unit": None,
                    "sensor_height_m": None,
                    "sensor_depth_m": None,
                    "aggregation": None,
                    "sampling_interval_s": None,
                    "latency_s": None,
                    "qaqc_level": None,
                    "metadata_retrieved_utc": retrieved,
                    "source_details_json": json.dumps(st_meta.get(key), ensure_ascii=False),
                })
    except Exception:
        pass

    # Deduplicate rows (same variable_native)
    if rows:
        df = pd.DataFrame(rows)
        df = df.drop_duplicates(subset=["station_uid", "variable_native"])
        return df.to_dict(orient="records")

    return rows


# -------------------------
# SNOTEL: variable availability via reportGenerator
# -------------------------

def snotel_reportgen_csv(site_id: str, element: str) -> Optional[pd.DataFrame]:
    """
    Fetch a small CSV from reportGenerator for one element. If element isn't supported, returns None.
    element examples: 'WTEQ' (SWE), 'SNWD' (snow depth), 'PRCP' (precip), 'TOBS' (air temp), etc.
    """
    # This URL pattern is finicky; we keep it minimal: daily values for a short window.
    # Station format in WCC: "SNOTEL:<id>_ID_SNTL" sometimes; but for customSingleStationReport we can use numeric id in the path.
    # We'll use a safe query approach: if it fails, return None.
    # Example path format used by WCC often includes state abbreviation; we keep generic & just detect success.
    try:
        # A short period to just see if data exists and column headers
        # YYYY-MM-DD is fine; use a short window
        start = "2025-01-01"
        end = "2025-01-10"
        # format: .../<stationId>:ID:SNTL|id=<site_id> ??? varies.
        # Practical approach: use the "station" syntax used in many WCC examples:
        station = f"{site_id}:ID:SNTL"
        url = f"{SNOTEL_REPORTGEN}/{station}|id={site_id}&report=Daily+Data"
        params = {
            "timeseries": element,
            "format": "csv",
            "sdate": start,
            "edate": end,
        }
        r = requests.get(url, params=params, timeout=60)
        if r.status_code != 200 or "Error" in r.text[:200]:
            return None

        from io import StringIO
        df = pd.read_csv(StringIO(r.text))
        return df
    except Exception:
        return None


def infer_snotel_variables(site_id: str) -> List[Dict[str, Any]]:
    """
    Infer which standard SNOTEL elements are available by probing a short reportGenerator window.
    This is slower than Synoptic but robust and reproducible.
    """
    candidates = [
        ("WTEQ", "SWE"),
        ("SNWD", "snow_depth"),
        ("PRCP", "precip"),
        ("TOBS", "air_temp"),
        # Add more if needed: soil temp/moisture vary by station
        ("SMS", "soil_moisture"),   # may not exist
        ("STO", "soil_temp"),       # may not exist
    ]

    retrieved = utcnow_iso()
    rows: List[Dict[str, Any]] = []

    for element, vcode in candidates:
        df = snotel_reportgen_csv(site_id, element)
        if df is None or df.empty:
            continue

        # If we got a dataframe, element exists; units often in header/metadata; keep raw columns as evidence
        rows.append({
            "station_uid": f"SNOTEL:{site_id}",
            "source": "SNOTEL",
            "variable_code": vcode,
            "variable_native": element,
            "unit": None,
            "sensor_height_m": None,
            "sensor_depth_m": None,
            "aggregation": "daily",
            "sampling_interval_s": 86400,
            "latency_s": None,
            "qaqc_level": None,
            "metadata_retrieved_utc": retrieved,
            "source_details_json": json.dumps({"columns": df.columns.tolist()}, ensure_ascii=False),
        })

    return rows


# -------------------------
# MAIN
# -------------------------

def main():
    stations = pd.read_csv(STATIONS_ALL_CSV)

    # Enriched stations output (we add retrieval timestamp and keep existing metadata)
    stations["metadata_retrieved_utc"] = utcnow_iso()
    stations.to_csv(OUT_STATIONS_ENRICHED, index=False)
    print(f"[OUT] {OUT_STATIONS_ENRICHED} ({len(stations)} rows)")

    var_rows: List[Dict[str, Any]] = []

    # SYNOPTIC variable inventory
    syn = stations[stations["source"] == "SYNOPTIC"].copy()
    for stid in syn["source_station_id"].dropna().astype(str).unique():
        stid = stid.strip()
        if not stid:
            continue
        var_rows.extend(parse_synoptic_variables(stid))
        time.sleep(0.1)  # be polite

    # SNOTEL variable inventory
    snot = stations[stations["source"] == "SNOTEL"].copy()
    for site_id in snot["source_station_id"].dropna().astype(str).unique():
        site_id = site_id.strip()
        if not site_id:
            continue
        var_rows.extend(infer_snotel_variables(site_id))
        time.sleep(0.2)

    vars_df = pd.DataFrame(var_rows)
    if not vars_df.empty:
        vars_df = vars_df.drop_duplicates(subset=["station_uid", "variable_native"])
    vars_df.to_csv(OUT_VARS, index=False)
    print(f"[OUT] {OUT_VARS} ({len(vars_df)} rows)")

    # Quick summaries
    if not vars_df.empty:
        print("\n[SUMMARY] Variables per source:")
        print(vars_df.groupby("source")["variable_native"].nunique())

        print("\n[SUMMARY] Stations with >=1 variable row:")
        print(vars_df.groupby("source")["station_uid"].nunique())


if __name__ == "__main__":
    main()

[OUT] out/stations_enriched.csv (1260 rows)
[OUT] out/station_variables.csv (10460 rows)

[SUMMARY] Variables per source:
source
SYNOPTIC    150
Name: variable_native, dtype: int64

[SUMMARY] Stations with >=1 variable row:
source
SYNOPTIC    1051
Name: station_uid, dtype: int64


In [5]:
import pandas as pd
vars_df = pd.read_csv("out/station_variables.csv")

# quante stazioni hanno almeno una variabile?
vars_df.groupby("source")["station_uid"].nunique()

# top variabili per Synoptic
vars_df[vars_df.source=="SYNOPTIC"]["variable_code"].value_counts().head(30)

# SNOTEL: quante stazioni hanno SWE?
vars_df[(vars_df.source=="SNOTEL") & (vars_df.variable_native=="WTEQ")]["station_uid"].nunique()

0

In [6]:
js = synoptic_station_metadata("COSI1")
list(js["STATION"][0].keys())

['ID',
 'STID',
 'NAME',
 'ELEVATION',
 'LATITUDE',
 'LONGITUDE',
 'STATUS',
 'MNET_ID',
 'STATE',
 'COUNTRY',
 'TIMEZONE',
 'ELEV_DEM',
 'PERIOD_OF_RECORD',
 'UNITS',
 'RESTRICTED',
 'RESTRICTED_METADATA']

In [7]:
import os, requests, json

token = os.getenv("SYNOPTIC_TOKEN","").strip()
url = "https://api.synopticdata.com/v2/stations/metadata"
params = {
    "stid": "COSI1",
    "token": token,
    "complete": 1,
    "sensorvars": 1,
}
js = requests.get(url, params=params, timeout=60).json()

print("SUMMARY:", js.get("SUMMARY", {}))
print("Top keys:", js.keys())
print("Station keys:", list(js.get("STATION",[{}])[0].keys())[:80])

SUMMARY: {'NUMBER_OF_OBJECTS': 1, 'RESPONSE_CODE': 1, 'RESPONSE_MESSAGE': 'OK', 'METADATA_QUERY_TIME': '2.5 ms', 'METADATA_PARSE_TIME': '0.4 ms', 'TOTAL_METADATA_TIME': '2.8 ms', 'TOTAL_TIME': '2.8 ms', 'VERSION': 'v2.31.0'}
Top keys: dict_keys(['STATION', 'SUMMARY'])
Station keys: ['ID', 'STID', 'NAME', 'ELEVATION', 'LATITUDE', 'LONGITUDE', 'STATUS', 'MNET_ID', 'STATE', 'COUNTRY', 'TIMEZONE', 'ELEV_DEM', 'NWSZONE', 'NWSFIREZONE', 'GACC', 'SHORTNAME', 'LONGNAME', 'URL', 'PROGRAM', 'CITATION', 'SGID', 'COUNTY', 'WIMS_ID', 'CWA', 'PERIOD_OF_RECORD', 'PROVIDERS', 'SITING', 'SENSOR_VARIABLES', 'UNITS', 'RESTRICTED', 'RESTRICTED_METADATA']


In [10]:
import json
import pandas as pd

def synoptic_metadata_complete(stid: str, token: str, timeout: int = 60) -> dict:
    url = "https://api.synopticdata.com/v2/stations/metadata"
    params = {"stid": stid, "token": token, "complete": 1, "sensorvars": 1}
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def parse_synoptic_station_complete(js: dict) -> pd.DataFrame:
    """Return 1-row stations df for a single station metadata payload."""
    st = (js.get("STATION") or [{}])[0]

    row = {
        "source": "SYNOPTIC",
        "source_station_id": st.get("STID"),
        "station_uid": f"SYNOPTIC:{st.get('STID')}" if st.get("STID") else None,
        "station_name": st.get("NAME"),
        "latitude": pd.to_numeric(st.get("LATITUDE"), errors="coerce"),
        "longitude": pd.to_numeric(st.get("LONGITUDE"), errors="coerce"),
        "elevation_m": pd.to_numeric(st.get("ELEVATION"), errors="coerce"),
        "mnet_id": st.get("MNET_ID"),
        "shortname": st.get("SHORTNAME"),
        "longname": st.get("LONGNAME"),
        "county": st.get("COUNTY"),
        "timezone": st.get("TIMEZONE"),
        "nwszone": st.get("NWSZONE"),
        "cwa": st.get("CWA"),
        "url": st.get("URL"),
        "program": st.get("PROGRAM"),
        "citation": st.get("CITATION"),
        "period_of_record_json": json.dumps(st.get("PERIOD_OF_RECORD"), ensure_ascii=False),
        "providers_json": json.dumps(st.get("PROVIDERS"), ensure_ascii=False),
        "siting_json": json.dumps(st.get("SITING"), ensure_ascii=False),
        "restricted": st.get("RESTRICTED"),
        "restricted_metadata": st.get("RESTRICTED_METADATA"),
    }
    return pd.DataFrame([row])

def _flatten_sensor_variables(sensor_vars):
    """
    Sensor variables can be dict or list depending on station/network.
    Normalize to list of (variable_name, payload_dict).
    """
    if sensor_vars is None:
        return []
    if isinstance(sensor_vars, dict):
        return list(sensor_vars.items())
    if isinstance(sensor_vars, list):
        out = []
        for item in sensor_vars:
            if isinstance(item, dict):
                # if item has a single key like {"air_temp": {...}}
                if len(item) == 1:
                    k = next(iter(item.keys()))
                    out.append((k, item.get(k)))
                else:
                    # otherwise store as anonymous block
                    out.append((item.get("variable") or item.get("name") or "unknown", item))
            else:
                out.append(("unknown", {"value": item}))
        return out
    # fallback
    return [("unknown", {"value": sensor_vars})]

def parse_synoptic_variables_complete(js: dict) -> pd.DataFrame:
    """
    Build long-format station_variables rows from SENSOR_VARIABLES + UNITS.

    Output is one row per *sensor variable* (e.g., air_temp_1), not per family (air_temp).
    """
    st = (js.get("STATION") or [{}])[0]
    stid = st.get("STID")
    station_uid = f"SYNOPTIC:{stid}" if stid else None

    units = st.get("UNITS") or {}
    sensor_vars = st.get("SENSOR_VARIABLES") or {}

    rows = []

    # sensor_vars is typically: { "air_temp": { "air_temp_1": {...}, ... }, "wind_speed": {...}, ... }
    for family, family_payload in sensor_vars.items():
        if family_payload is None:
            continue

        # Some families may be directly a dict of sensors; otherwise store raw
        if isinstance(family_payload, dict):
            for sensor_key, sensor_payload in family_payload.items():
                sensor_payload = sensor_payload or {}

                # Unit is often keyed by sensor_key in UNITS (e.g., "air_temp_1")
                unit = units.get(sensor_key) or units.get(family)

                # Height/depth: keys can vary; pull common patterns
                height = (
                    sensor_payload.get("height")
                    or sensor_payload.get("HEIGHT")
                    or sensor_payload.get("sensor_height")
                    or sensor_payload.get("SENSOR_HEIGHT")
                )
                depth = (
                    sensor_payload.get("depth")
                    or sensor_payload.get("DEPTH")
                    or sensor_payload.get("sensor_depth")
                    or sensor_payload.get("SENSOR_DEPTH")
                )

                # Period of record sometimes exists per sensor, sometimes only station-level
                por = (
                    sensor_payload.get("PERIOD_OF_RECORD")
                    or sensor_payload.get("period_of_record")
                    or None
                )

                rows.append({
                    "station_uid": station_uid,
                    "source": "SYNOPTIC",
                    "source_station_id": stid,
                    "variable_family": family,          # e.g., air_temp
                    "variable_native": sensor_key,       # e.g., air_temp_1
                    "variable_code": family,             # keep family as normalized code for now
                    "unit": unit,
                    "sensor_height_m": pd.to_numeric(height, errors="coerce") if height is not None else None,
                    "sensor_depth_m": pd.to_numeric(depth, errors="coerce") if depth is not None else None,
                    "position": sensor_payload.get("position"),
                    "dimension": sensor_payload.get("dimension"),
                    "period_of_record_json": json.dumps(por, ensure_ascii=False) if por is not None else None,
                    "source_details_json": json.dumps(sensor_payload, ensure_ascii=False),
                })
        else:
            # unexpected structure: still keep traceability
            rows.append({
                "station_uid": station_uid,
                "source": "SYNOPTIC",
                "source_station_id": stid,
                "variable_family": family,
                "variable_native": family,
                "variable_code": family,
                "unit": units.get(family),
                "sensor_height_m": None,
                "sensor_depth_m": None,
                "position": None,
                "dimension": None,
                "period_of_record_json": None,
                "source_details_json": json.dumps(family_payload, ensure_ascii=False),
            })

    return pd.DataFrame(rows)



In [11]:
token = os.getenv("SYNOPTIC_TOKEN","").strip()
js = synoptic_metadata_complete("COSI1", token)

df_station = parse_synoptic_station_complete(js)
df_vars = parse_synoptic_variables_complete(js)

df_station
df_vars.head(20)

,station_uid,source,source_station_id,variable_family,variable_native,variable_code,unit,sensor_height_m,sensor_depth_m,position,dimension,period_of_record_json,source_details_json
0,SYNOPTIC:COSI1,SYNOPTIC,COSI1,air_temp,air_temp_1,air_temp,None,None,None,None,{},"{""start"": ""2023-05-24T18:12:00Z"", ""end"": ""2026...","{""position"": null, ""dimension"": {}, ""summary"":..."
1,SYNOPTIC:COSI1,SYNOPTIC,COSI1,snow_depth,snow_depth_1,snow_depth,None,None,None,None,{},"{""start"": ""2023-05-24T18:12:00Z"", ""end"": ""2026...","{""position"": null, ""dimension"": {}, ""summary"":..."
2,SYNOPTIC:COSI1,SYNOPTIC,COSI1,soil_temp,soil_temp_1,soil_temp,None,None,None,None,{},"{""start"": ""2025-12-18T17:00:00Z"", ""end"": ""2026...","{""position"": null, ""dimension"": {}, ""summary"":..."
3,SYNOPTIC:COSI1,SYNOPTIC,COSI1,precip_accum,precip_accum_1,precip_accum,None,None,None,None,{},"{""start"": ""2023-05-24T18:12:00Z"", ""end"": ""2026...","{""position"": null, ""dimension"": {}, ""summary"":..."
4,SYNOPTIC:COSI1,SYNOPTIC,COSI1,precip_smoothed,precip_smoothed_1,precip_smoothed,None,None,None,None,{},"{""start"": null, ""end"": null}","{""position"": null, ""dimension"": {}, ""summary"":..."
5,SYNOPTIC:COSI1,SYNOPTIC,COSI1,snow_smoothed,snow_smoothed_1,snow_smoothed,None,None,None,None,{},"{""start"": null, ""end"": null}","{""position"": null, ""dimension"": {}, ""summary"":..."
6,SYNOPTIC:COSI1,SYNOPTIC,COSI1,snow_water_equiv,snow_water_equiv_1,snow_water_equiv,None,None,None,None,{},"{""start"": ""2023-05-24T18:12:00Z"", ""end"": ""2026...","{""position"": null, ""dimension"": {}, ""summary"":..."


In [12]:
df_vars = parse_synoptic_variables_complete(js)
df_vars[["variable_family","variable_native","unit","sensor_height_m","dimension","position"]].head(30)


,variable_family,variable_native,unit,sensor_height_m,dimension,position
0,air_temp,air_temp_1,None,None,{},None
1,snow_depth,snow_depth_1,None,None,{},None
2,soil_temp,soil_temp_1,None,None,{},None
3,precip_accum,precip_accum_1,None,None,{},None
4,precip_smoothed,precip_smoothed_1,None,None,{},None
5,snow_smoothed,snow_smoothed_1,None,None,{},None
6,snow_water_equiv,snow_water_equiv_1,None,None,{},None


In [13]:
st = js["STATION"][0]
units = st.get("UNITS", {})
print("UNITS type:", type(units))
print("UNITS keys sample:", list(units.keys())[:50])
print("UNITS content:", units)

UNITS type: <class 'dict'>
UNITS keys sample: ['position', 'elevation']
UNITS content: {'position': 'm', 'elevation': 'ft'}


In [14]:
import os, requests
token = os.getenv("SYNOPTIC_TOKEN","").strip()
url = "https://api.synopticdata.com/v2/stations/latest"
jsL = requests.get(url, params={"stid":"COSI1","token":token}, timeout=60).json()

stL = jsL["STATION"][0]
print("LATEST UNITS keys sample:", list((stL.get("UNITS") or {}).keys())[:50])
print("LATEST OBS keys sample:", list((stL.get("OBSERVATIONS") or {}).keys())[:50])

LATEST UNITS keys sample: ['position', 'elevation']
LATEST OBS keys sample: ['air_temp_value_1', 'snow_depth_value_1', 'soil_temp_value_1', 'precip_accum_value_1', 'snow_water_equiv_value_1']


In [15]:
def enrich_units_from_latest(df_vars: pd.DataFrame, js_latest: dict) -> pd.DataFrame:
    st = js_latest["STATION"][0]
    units = st.get("UNITS") or {}
    out = df_vars.copy()

    # mapping: variable_native (e.g. air_temp_1) -> try common latest keys
    def guess_unit(var_native: str, family: str):
        # common latest patterns
        candidates = [
            f"{family}_value_1",
            f"{var_native}_value_1",  # sometimes matches
            family,
            var_native,
        ]
        for k in candidates:
            if k in units:
                return units[k]
        return None

    out["unit"] = out.apply(lambda r: guess_unit(r["variable_native"], r["variable_family"]), axis=1)
    return out

In [16]:
def enrich_units_from_latest(df_vars: pd.DataFrame, js_latest: dict) -> pd.DataFrame:
    st = js_latest["STATION"][0]
    units = st.get("UNITS") or {}
    out = df_vars.copy()

    # mapping: variable_native (e.g. air_temp_1) -> try common latest keys
    def guess_unit(var_native: str, family: str):
        # common latest patterns
        candidates = [
            f"{family}_value_1",
            f"{var_native}_value_1",  # sometimes matches
            family,
            var_native,
        ]
        for k in candidates:
            if k in units:
                return units[k]
        return None

    out["unit"] = out.apply(lambda r: guess_unit(r["variable_native"], r["variable_family"]), axis=1)
    return out

In [17]:
df_vars2 = enrich_units_from_latest(df_vars, jsL)
df_vars2[["variable_family","variable_native","unit"]]

,variable_family,variable_native,unit
0,air_temp,air_temp_1,None
1,snow_depth,snow_depth_1,None
2,soil_temp,soil_temp_1,None
3,precip_accum,precip_accum_1,None
4,precip_smoothed,precip_smoothed_1,None
5,snow_smoothed,snow_smoothed_1,None
6,snow_water_equiv,snow_water_equiv_1,None


In [18]:
print(st.get("UNITS"))
print(stL.get("UNITS"))

{'position': 'm', 'elevation': 'ft'}
{'position': 'm', 'elevation': 'ft'}
